### What is a Transformer?
A **Transformer** is a neural network architecture based primarily on the **self-attention mechanism**. It processes relationships between tokens in a sequence, enabling the model to understand and generate contextual text efficiently.

* **Key Difference:** Instead of processing text strictly one word at a time like older sequential models (RNNs/LSTMs), a Transformer analyzes the relationships between all tokens simultaneously.

###  Internal Working Pipeline:
$$\text{Token} \rightarrow \text{Embedding} \rightarrow \text{Positional Encoding} \rightarrow \text{Self-Attention} \rightarrow \text{FFN (Feed-Forward Network)} \rightarrow \text{Final Output}$$

### Why did we need Transformers?
Before Transformers, NLP commonly used **RNNs, LSTMs, and GRUs**.

* **The Problem:**
  * For example, in the sentence: *"The dog chased the cat because it was running..."*
  * To understand what **"it"** refers to, the model needs information from other words.
  * **Sequential Bottleneck:** RNNs/LSTMs process text sequentially (one by one), making them extremely hard to parallelize during training.
  * **Information Decay:** Information tends to weaken over long sequences (**long-range dependency problem**).
* **The Solution:** The Transformer introduced **Self-Attention**, which allows it to directly calculate relationships between any two words, regardless of their distance in the text.


###  High-Level Transformer Pipeline:
$$\text{User Text} \rightarrow \text{Tokenization} \rightarrow \text{Token IDs} \rightarrow \text{Token Embeddings} \rightarrow \text{Positional Information} \rightarrow \text{Transformer Blocks} \rightarrow \text{Linear Layer} \rightarrow \text{Logits} \rightarrow \text{Softmax} \rightarrow \text{Next Token Probability}$$

1. **Input Text:** The model cannot directly understand raw text sentences.
2. **Tokenizer:** Converts raw text into smaller units called **tokens**.
   * *Example:* `"What is Machine Learning"` $\rightarrow$ `["What", "is", "Machine", "Learning"]` $\rightarrow$ Mapped to unique integer **Token IDs** (e.g., `[1, 20, 34, 45]`).
3. **Token Embedding:** Token IDs themselves have no semantic meaning. The model uses an **embedding matrix** to convert discrete Token IDs into dense numerical vectors that capture semantic relationships.

4. **Positional Information:** The self-attention mechanism does not inherently recognize the order of words.
   * *Example:* `"Dog bites man"` and `"Man bites dog"` contain identical words but different meanings.
   * **Traditional Models:** Use static **Sinusoidal Positional Encoding** ($Token\,Embedding + Positional\,Encoding$).
   * **Modern LLMs:** Use learned positional encodings or **Rotary Positional Embedding (RoPE)**.

5. **Self-Attention ($Q, K, V$):** Allows each token to focus on other relevant tokens. For every token, the model creates **Query (Q)**, **Key (K)**, and **Value (V)** vectors using trained weight matrices:
   * $\text{Query (Q)} = \text{Input Embedding} \times W_q$ (What information am I looking for?)
   * $\text{Key (K)} = \text{Input Embedding} \times W_k$ (What topic/context do I represent?)
   * $\text{Value (V)} = \text{Input Embedding} \times W_v$ (The actual content or information of the token.)



### Mathematical Formula:
$$\text{Attention}(Q,K,V)=\text{Softmax}\left(\frac{QK^{T}}{\sqrt{d_{k}}}\right)V$$

* **$QK^T$**: Computes raw similarity scores (attention weights) between all tokens.
* **Why divide by $\sqrt{d_{k}}$?** ($d_k$ is the dimension of Key vectors). Without this scaling, dot-product values can grow extremely large, pushing the Softmax function into regions with **extremely small gradients** (vanishing gradient problem). This scaling ensures stable training.
* **Softmax:** Converts attention scores into a probability distribution summing up to 1.
* **Multi-Head Attention (MHA):** Instead of running attention once, the model runs multiple attention heads in parallel. Different heads focus on different aspects:
  * *Head 1:* Grammatical relationships.
  * *Head 2:* Subject-Object relationships.
  * *Head 3:* Positional relationships.
  * *Head 4:* Semantic context.


* **Residual (Skip) Connection:** $\text{Output} = \text{Input} + \text{Attention}(\text{Input})$. This allows gradients to flow smoothly during backpropagation, enabling the training of incredibly deep networks.
* **Layer Normalization (LayerNorm):** Stabilizes activations.
  * *Post-LN:* Applied after sub-layers (classic Transformer).
  * *Pre-LN:* Applied before sub-layers (used in modern LLMs for training stability).
* **Feed-Forward Network (FFN):** Processes each token individually and non-linearly after attention has integrated the contextual information.


### Encoder vs. Decoder:
* **Encoder-Only:** Great for understanding tasks (e.g., BERT for classification, NER, embeddings).
* **Decoder-Only:** Great for generative tasks (e.g., GPT, Llama, and most modern LLMs).

### Causal Attention / Causal Masking:
In decoder-only autoregressive models, a token is **not allowed** to look at future tokens.
* *Example:* While predicting `"Machine"` in `"I love Machine Learning"`, the model can only look at `"I"` and `"love"`. It masks out `"Learning"` to prevent information leakage during training.

### Training Loss:
LLMs are trained on **Next Token Prediction** using **Cross-Entropy Loss**:
$$L=-\sum _{i=1}^{N}Y_{i}\log (P_{i})$$



* **Temperature:** Controls randomness. Low temperature = deterministic/focused; High temperature = creative/diverse.
* **Top-K:** Considers only the $K$ most probable next tokens.
* **Top-P (Nucleus Sampling):** Considers the smallest set of tokens whose cumulative probability exceeds threshold $P$.
* **KV Cache:** Stores previous Key ($K$) and Value ($V$) tensors in GPU memory to avoid expensive recomputations during token-by-token generation.
* **Context Window:** The maximum limit of tokens a model can process at once.

---

##  Advanced QA (Interview Prep)

### **Q14. Why is Transformer inference expensive?**
1. **Autoregressive Generation:** Generating 100 tokens requires running the entire network 100 times sequentially.
2. **Memory Bottleneck (IO Bound):** At every step, the GPU must fetch massive model weights and previous KV caches from High Bandwidth Memory (HBM) to its SRAM, making memory bandwidth the primary speed limit.

### **Q18. How would you reduce LLM inference latency?**
* **KV Caching:** Prevents redundant attention calculations.
* **Quantization:** Reduces weights precision (e.g., FP16 to INT8 or INT4) to save memory and speed up computation.
* **Speculative Decoding:** Uses a small draft model to generate tokens quickly, which are validated in one parallel pass by the large target LLM.
* **Optimized Engines:** Using frameworks like **vLLM**, **TensorRT-LLM**, or **TGI**.

### **Q19. What is KV Cache Optimization?**
Standard KV caches scale linearly with context size and take up huge GPU memory.
* **PagedAttention (vLLM):** Allocates memory for KV Cache in non-contiguous virtual blocks, drastically reducing fragmentation and memory waste.
* **MQA / GQA (Multi/Grouped Query Attention):** Shares Key and Value heads across multiple Query heads, significantly shrinking the KV cache memory footprint.

### **Q20. What causes high GPU memory usage in a Transformer?**
1. **Model Weights:** Static memory required to load billions of parameters.
2. **KV Cache:** Dynamic memory that scales rapidly with sequence length and batch size.
3. **Activation Memory:** Temporary intermediate tensors stored during training/inference forward passes.

### **Q21. How does context length affect latency?**
* **Prefill Phase:** Processing the initial input prompt scales quadratically ($O(N^2)$) with prompt length.
* **Decoding Phase:** A longer context results in a larger KV Cache, requiring more GPU memory bandwidth to read the cache at each step.

### **Q22. How does batching improve inference throughput?**
Batching processes multiple requests simultaneously.
* **Efficiency:** Instead of loading the massive model weights into GPU memory for a single user, the same loaded weights are reused to process queries for multiple users at once.
* **Continuous Batching:** Dynamically schedules new requests at the token level as older requests finish, keeping the GPU highly utilized without idle time.